In [6]:
!pip install requests qrcode pillow pandas

In [2]:
import os
import json
import requests
import pandas as pd
import qrcode
from pathlib import Path

# --- CONFIGURATION ---
GAS_URL = "https://script.google.com/macros/s/AKfycbyqfQGnUMg4T4zQLt5WHo6WYrGjVHaTwWg5HhJlapXm16CwxL2aGRM5IbBtAdCEvJpckw/exec"
SHEET_FILENAME = "guests_data.csv"
OUTPUT_FOLDER = "QR code data"

In [3]:
# Create the output directory if it doesn't already exist
output_dir = Path(OUTPUT_FOLDER)
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Directory ready: {output_dir.resolve()}")

Directory ready: /mnt/data/Portfolio Projects/Scanner-App.Seif-Lashin.io/QR code data


In [4]:
# Check if dataset is cached locally; if not, fetch from Google Apps Script
local_file = Path(SHEET_FILENAME)

if not local_file.exists():
    print("Downloading dataset from Google Apps Script...")
    # Add getGuests action param if your script requires it
    target_url = f"{GAS_URL}?action=getGuests" if "?" not in GAS_URL else GAS_URL
    
    response = requests.get(target_url)
    response.raise_for_status()
    
    data = response.json()
    
    # Save as CSV for local persistence
    df = pd.DataFrame(data)
    df.to_csv(local_file, index=False)
    print(f"Downloaded and saved to {SHEET_FILENAME}")
else:
    print(f"Found local file '{SHEET_FILENAME}'. Loading from disk...")
    df = pd.read_csv(local_file)

print(f"Total guests loaded: {len(df)}")
df.head()

Downloaded and saved to guests_data.csv
Total guests loaded: 10


,uuid,name,email,attended,timestamp
0,USR-9821-A,Seif Lashin,seif.lashin@example.com,True,2026-08-24T14:56:34.102Z
1,USR-1042-B,Omar Hassan,omar.hassan@example.com,True,2026-08-25T07:14:46.944Z
2,USR-3391-C,Nour El-Din,nour.eldin@example.com,False,
3,USR-4820-D,Kareem Zaki,kareem.zaki@example.com,False,
4,USR-5193-E,Mariam Aly,mariam.aly@example.com,False,


In [5]:
# Ensure UUID column exists (case-insensitive check)
uuid_col = next((col for col in df.columns if col.lower() == 'uuid'), None)

if not uuid_col:
    raise KeyError(f"Could not find a 'uuid' column in sheet. Found: {list(df.columns)}")

generated_count = 0

for index, row in df.iterrows():
    raw_uuid = str(row[uuid_col]).strip()
    
    # Skip empty or missing UUID entries
    if not raw_uuid or raw_uuid.lower() == 'nan':
        continue
    
    # Generate QR Code encoding only the raw UUID string
    qr = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_M,
        box_size=10,
        border=4,
    )
    qr.add_data(raw_uuid)
    qr.make(fit=True)
    
    img = qr.make_image(fill_color="black", back_color="white")
    
    # Save image named {UUID}.png inside the folder
    file_path = output_dir / f"{raw_uuid}.png"
    img.save(file_path)
    generated_count += 1

print(f"Successfully generated {generated_count} QR codes in '{OUTPUT_FOLDER}' folder.")

Successfully generated 10 QR codes in 'QR code data' folder.
